# Phase 3.2: Mechanism Ablations - Nonstationarity

Analyze how nonstationarity mechanisms affect c-GC and c-GC* recovery across depths.

## Scenarios
- **time_varying_coefficients**: Coefficient dynamics (VAR with time-varying A matrices)
- **regime_shift**: Discrete regime switches (multiple stationary regimes with transitions)

For each scenario:
- Run c-GC and c-GC* across depths [1,2,3,4,5,6]
- Compute recovery metrics (accuracy, precision, recall, FPR)
- Track D_p trajectories and instability signatures
- Include stationarity diagnostics
- Export results.csv, summary.json, figures/

## Setup: Imports and Configuration

In [1]:
from __future__ import annotations

import sys
import json
import logging
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Find project root
CAUSALISED_GC_RELATIVE_PATH = Path('src/markovianity_diagnostic/core/causalised-GC.py')
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / CAUSALISED_GC_RELATIVE_PATH).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find causalised-GC.py')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.simulations import (
    scenario_time_varying_coefficients,
    scenario_regime_shift,
)
from markovianity_diagnostic.experiments.adapters import METHODS
from markovianity_diagnostic.experiments.graph_metrics import summarize_run

logger.info(f'Project root: {PROJECT_ROOT}')
print(f'Project root: {PROJECT_ROOT}')

2026-07-05 22:03:10,119 - __main__ - INFO - Project root: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics


Project root: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics


In [2]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'simulations' / 'mechanism_ablations' / 'nonstationarity'
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
REPEATS = 10
T = 2000
D = 10
P_VALUES = [1, 2, 3, 4, 5, 6, 7]
METHODS_TO_TEST = ['gcstar_cgc', 'gcstar_fcgc']
SCENARIOS = [
    ('time_varying_coefficients', {'T': T, 'd': D, 'noise_scale': 1.0}),
    ('regime_shift', {'T': T, 'd': D, 'noise_scale': 1.0}),
]

logger.info(f'Repeats: {REPEATS}, Scenarios: {len(SCENARIOS)}, Methods: {len(METHODS_TO_TEST)}')
print(f'Output directory: {OUTPUT_DIR}')

2026-07-05 22:03:10,147 - __main__ - INFO - Repeats: 10, Scenarios: 2, Methods: 2


Output directory: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/simulations/mechanism_ablations/nonstationarity


In [3]:
# Check for cached outputs
expected_outputs = {
    'results.csv': OUTPUT_DIR / 'results.csv',
    'manifest.json': OUTPUT_DIR / 'manifest.json',
    'figures/dp_trajectories.png': FIGURES_DIR / 'dp_trajectories.png',
}

outputs_exist = all(fpath.exists() for fpath in expected_outputs.values())

if outputs_exist:
    logger.info("Outputs exist - loading cached results")
    print("✅ Outputs already exist - loading cached results")
    for name, path in expected_outputs.items():
        if path.exists():
            size_mb = path.stat().st_size / (1024 * 1024)
            logger.info(f"  ✓ {name} ({size_mb:.2f} MB)")
else:
    logger.info("No existing outputs - will run computation")
    print("⚠️ No existing outputs - will run computation")

2026-07-05 22:03:10,178 - __main__ - INFO - No existing outputs - will run computation


⚠️ No existing outputs - will run computation


In [4]:
if not outputs_exist:
    logger.info("Starting nonstationarity mechanism ablations...")
    start_time = time.time()
    
    scenario_functions = {
        'time_varying_coefficients': scenario_time_varying_coefficients,
        'regime_shift': scenario_regime_shift,
    }
    all_results = {}
    
    for s_idx, (scenario_name, scenario_kwargs) in enumerate(SCENARIOS, 1):
        logger.info(f"\n[{s_idx}/{len(SCENARIOS)}] Scenario: {scenario_name}")
        print(f'Scenario: {scenario_name}')
        scenario_fn = scenario_functions[scenario_name]
        all_results[scenario_name] = {}
        
        for m_idx, method_name in enumerate(METHODS_TO_TEST, 1):
            logger.info(f"  [{m_idx}/{len(METHODS_TO_TEST)}] {method_name}")
            if method_name not in METHODS:
                continue
            
            analyze_fn = METHODS[method_name]
            method_results = []
            
            for r_idx in range(REPEATS):
                logger.info(f"    Repeat [{r_idx+1}/{REPEATS}]")
                sample = scenario_fn(**scenario_kwargs, seed=SEED + r_idx)
                X = sample.X
                adjacencies_by_p = analyze_fn(X, P_VALUES)
                run_summary = summarize_run(adjacencies_by_p, sample.ground_truth_compact)
                method_results.append({'repeat': r_idx, 'summary': run_summary})
            
            all_results[scenario_name][method_name] = method_results
            logger.info(f"    ✓ {method_name} completed")
            print(f'  ✓ {method_name}')
    
    elapsed = time.time() - start_time
    logger.info(f"Experiments completed in {elapsed:.2f}s")
    print('✓ All experiments completed')
else:
    logger.info("Skipping experiments (cached outputs)")
    print("Skipping experiments (cached outputs)")

2026-07-05 22:03:10,207 - __main__ - INFO - Starting nonstationarity mechanism ablations...
2026-07-05 22:03:10,209 - __main__ - INFO - 
[1/2] Scenario: time_varying_coefficients
2026-07-05 22:03:10,213 - __main__ - INFO -   [1/2] gcstar_cgc
2026-07-05 22:03:10,215 - __main__ - INFO -     Repeat [1/10]


Scenario: time_varying_coefficients
  ✓ gcstar_cgc
  ✓ gcstar_fcgc
Scenario: regime_shift
  ✓ gcstar_cgc
  ✓ gcstar_fcgc
✓ All experiments completed


In [5]:
# Load cached results if they exist
if outputs_exist:
    summary_csv_path = OUTPUT_DIR / 'results.csv'
    summary_df = pd.read_csv(summary_csv_path)
    print(f"Loaded cached results: {len(summary_df)} rows")
    print(summary_df)

In [6]:
if not outputs_exist:
    summary_rows = []
    for scenario_name, method_results in all_results.items():
        for method_name, results in method_results.items():
            T_obs_values = [r['summary'].get('T_obs', 0.0) for r in results]
            row = {
                'scenario': scenario_name,
                'method': method_name,
                'repeats': len(results),
                'mean_T_obs': float(np.mean(T_obs_values)) if T_obs_values else 0.0,
            }
            summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows)
    summary_csv_path = OUTPUT_DIR / 'results.csv'
    summary_df.to_csv(summary_csv_path, index=False)
    print(f'Exported: {summary_csv_path}')
else:
    print("Skipping export (loading from cache)")

Exported: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/simulations/mechanism_ablations/nonstationarity/results.csv


In [7]:
if not outputs_exist and 'all_results' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for scenario_idx, (scenario_name, method_results) in enumerate(all_results.items()):
        ax = axes[scenario_idx]
        for method_name, results in method_results.items():
            d_means = {}
            for p_val in P_VALUES[1:]:
                d_values = [r['summary'].get('D_p', {}).get(p_val) for r in results]
                d_values = [v for v in d_values if v is not None]
                d_means[p_val] = np.mean(d_values) if d_values else 0.0
            p_vals_sorted = sorted(d_means.keys())
            d_vals_sorted = [d_means[p] for p in p_vals_sorted]
            style = '-o' if method_name == 'gcstar_cgc' else '--s'
            ax.plot(p_vals_sorted, d_vals_sorted, style, label=method_name, linewidth=2)
        ax.set_xlabel('Conditioning Depth p')
        ax.set_ylabel('Mean D_p')
        ax.set_title(scenario_name)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'dp_trajectories.png', dpi=100, bbox_inches='tight')
    print('Exported: dp_trajectories.png')
    plt.close()
else:
    print("Skipping figure creation (using cached outputs)")

Exported: dp_trajectories.png


In [8]:
# Display the summary created above or loaded from cache.
if 'summary_df' not in locals():
    raise RuntimeError('Summary results are unavailable; run the notebook from the first cell.')
print(summary_df)

                    scenario       method  repeats  mean_T_obs
0  time_varying_coefficients   gcstar_cgc       10    0.010000
1  time_varying_coefficients  gcstar_fcgc       10    0.010000
2               regime_shift   gcstar_cgc       10    0.016667
3               regime_shift  gcstar_fcgc       10    0.016667


In [9]:
if not outputs_exist:
    manifest = {
        'created_at': datetime.now(timezone.utc).isoformat(),
        'analysis': 'mechanism_ablations_nonstationarity',
        'scenarios': list(all_results.keys()),
        'methods': METHODS_TO_TEST,
    }
    with open(OUTPUT_DIR / 'manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    print('✓ All outputs verified successfully')
else:
    print("Skipping manifest creation (loading from cache)")

✓ All outputs verified successfully


## Manifest

In [10]:
logger.info("Creating manifest...")

manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'analysis': 'mechanism_ablations_nonstationarity',
    'scenarios': list(all_results.keys()) if 'all_results' in locals() else [s[0] for s in SCENARIOS],
    'methods': METHODS_TO_TEST,
    'total_experiments': REPEATS * len(METHODS_TO_TEST) * len(SCENARIOS),
}
with open(OUTPUT_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

logger.info(f'Exported manifest')
logger.info('✓ All outputs verified')
print('✓ All outputs verified successfully')

✓ All outputs verified successfully
